# 🧠 Decoder-Only Transformer: A Complete Internal Tour
### Using `microsoft/Phi-3-mini-4k-instruct` in Google Colab

---

**What you will learn in this notebook:**

```
Raw Text
   ↓  Tokenization
input_ids  (integers)
   ↓  Embedding + Transformer Layers
Hidden States  (contextual vectors)
   ↓  LM Head (linear projection)
Logits  (scores over vocabulary)
   ↓  Decoding Strategy
Next Token
   ↓  Repeat until EOS
Final Text Output
```

Every step is broken into individual runnable cells with detailed explanations.
No black boxes. No shortcuts.

> ⚡ **Runtime tip:** Go to `Runtime → Change runtime type → T4 GPU` before running.


---
##  Install Dependencies

Three libraries power everything in this notebook:

| Library | Role |
|---|---|
| `transformers` | Hugging Face library — loads pretrained models, tokenizers, and pipelines |
| `torch` | PyTorch — the tensor computation engine behind every model operation |
| `accelerate` | Hugging Face utility that handles device placement (CPU/GPU/multi-GPU) transparently |

We pin versions to avoid breaking changes in this tutorial.


In [ ]:

!pip uninstall -y transformers
!pip install -q "transformers==4.45.2" torch accelerate





---
##  Import Libraries

- `torch` — everything tensor-related lives here
- `AutoTokenizer` — loads the right tokenizer for any model automatically
- `AutoModelForCausalLM` — loads a causal (decoder-only) language model
- `pipeline` — Hugging Face's high-level API that bundles tokenizer + model + decoding


In [ ]:
#  Import Libraries ───────────────────────────────────────────────────

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Verify GPU availability — Phi-3 is large; GPU is strongly recommended
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found. Inference will be slow. Switch to T4 runtime.")


---
## Load Model & Tokenizer

### What is a Tokenizer?
A tokenizer converts raw text into integer token IDs that the model understands.
Phi-3 uses a **SentencePiece / tiktoken-style** byte-pair encoding (BPE) tokenizer with a vocabulary of ~32,000 tokens.

```
"Hello world"  →  [15043, 3186]   (token IDs)
```

### What is `AutoModelForCausalLM`?
This loads a **causal** (left-to-right, decoder-only) language model.
"Causal" means each token can only attend to **previous** tokens — not future ones.
This is enforced via a **causal attention mask** (upper triangular mask set to −∞).

### What does `trust_remote_code=True` mean?
Some models (including Phi-3) ship custom Python code alongside their weights on the Hub.
Setting `trust_remote_code=True` tells Transformers: *"download and execute that custom code."*
Only do this for models you trust (e.g., official Microsoft releases).

### Memory note
We load in **4-bit quantization** (`load_in_4bit=True`) to fit Phi-3 (3.8B params) on a free T4 GPU.
Without quantization you'd need ~8 GB VRAM for fp16.


In [ ]:

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

print(f"Loading tokenizer from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True   # Phi-3 ships custom tokenizer code on the Hub
)

print(f"Tokenizer vocab size: {tokenizer.vocab_size:,} tokens")
print(f"EOS token: '{tokenizer.eos_token}'  (id={tokenizer.eos_token_id})")
print(f"PAD token: '{tokenizer.pad_token}'")

print("\nLoading model (this downloads ~2–4 GB, one-time)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,   # fp16 halves VRAM vs fp32
    device_map="auto",           # accelerate auto-places layers on GPU/CPU
)
model.eval()  # Disable dropout — we're doing inference, not training

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded!")
print(f"   Total parameters: {total_params/1e9:.2f} Billion")
print(f"   Model type: {type(model).__name__}")


---
##  Tokenization Step

### What are PyTorch Tensors?
A **tensor** is a multi-dimensional array — the core data structure in PyTorch.
Think of it like a NumPy array but with GPU support and automatic differentiation.

```
Scalar (0D tensor):  torch.tensor(42)
Vector (1D tensor):  torch.tensor([1, 2, 3])
Matrix (2D tensor):  torch.tensor([[1, 2], [3, 4]])
```

### `input_ids`
Shape: `[batch_size, sequence_length]`
Contains the integer ID of each token. The model looks up these IDs in an embedding table.

### `attention_mask`
Shape: `[batch_size, sequence_length]`
A binary tensor: `1` = this token is real, `0` = padding (ignore me).
For a single prompt with no padding, it's all `1`s.


In [ ]:

prompt = "The capital of France is"

# Tokenize: text → tensors
# return_tensors="pt" means "return PyTorch tensors" (vs "tf" for TensorFlow)
inputs = tokenizer(prompt, return_tensors="pt")

input_ids     = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

print("=" * 55)
print(f"Original prompt:  \"{prompt}\"")
print("=" * 55)

print(f"\n📐 input_ids shape:      {input_ids.shape}")
print(   "   Interpretation:       [batch_size=1, seq_len]")
print(f"   Raw tensor:            {input_ids}")

print(f"\n📐 attention_mask shape: {attention_mask.shape}")
print(f"   Raw tensor:            {attention_mask}")
print( "   (all 1s = no padding tokens in this single prompt)")

print("\n🔍 Token-by-token breakdown:")
print(f"  {'Token ID':>10}  Token String")
print(f"  {'─'*10}  {'─'*20}")
for token_id in input_ids[0]:
    token_str = tokenizer.decode([token_id.item()])
    print(f"  {token_id.item():>10}  '{token_str}'")

seq_len = input_ids.shape[1]
print(f"\n📊 Sequence length: {seq_len} tokens")


---
## Manual Forward Pass (Transformer Body)

### The Transformer stack: `model.model`
Phi-3 follows a standard decoder-only architecture:

```
input_ids
   ↓
[Embedding Layer]           ← token IDs → dense vectors
   ↓
[RoPE Positional Encoding]  ← injects position info into attention
   ↓
[Transformer Block 0]       ┐
[Transformer Block 1]       │  32 blocks in Phi-3-mini
      ...                   │  Each block = Grouped-Query Attention + MLP
[Transformer Block 31]      ┘
   ↓
hidden_states               ← shape: [batch, seq_len, hidden_dim]
```

### What are hidden states?
After passing through all transformer layers, each token position has a **hidden state** — a dense vector of `hidden_dim` floating-point numbers.
This vector encodes the **meaning** of that token **in context** of all preceding tokens.

The last token's hidden state is the model's "summary" of the entire input so far.


In [ ]:

# torch.no_grad() disables gradient tracking — critical for inference
# (saves memory and speeds things up; we're not doing backprop)
with torch.no_grad():
    # model.model = the transformer backbone (everything EXCEPT the final LM head)
    # output_hidden_states=True returns activations from every layer
    transformer_outputs = model.model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
    )

# The final hidden state: shape [batch_size, seq_len, hidden_dim]
hidden_states = transformer_outputs.last_hidden_state
all_hidden_states = transformer_outputs.hidden_states  # tuple: one per layer

batch_size, seq_len, hidden_dim = hidden_states.shape

print("=" * 55)
print("Transformer Body Output")
print("=" * 55)

print(f"\n📐 hidden_states.shape = {hidden_states.shape}")
print( "   Breakdown:")
print(f"     batch_size = {batch_size}   (we sent 1 prompt)")
print(f"     seq_len    = {seq_len}   (number of input tokens)")
print(f"     hidden_dim = {hidden_dim}  (Phi-3 embedding dimension)")

print(f"\n🧱 Number of transformer layers: {len(all_hidden_states)}")
print( "   (includes embedding layer output + 32 transformer block outputs)")

print(f"\n🔍 Last token hidden state (first 8 values):")
last_token_hidden = hidden_states[0, -1, :]  # shape: [hidden_dim]
print(f"   {last_token_hidden[:8].tolist()}")
print(f"   ... (and {hidden_dim - 8} more values)")
print( "   This vector is the model's internal 'understanding' of the prompt so far.")

print(f"\n📊 Data type: {hidden_states.dtype}  (fp16 = 2 bytes per value)")
print(f"   Memory used: {hidden_states.element_size() * hidden_states.nelement() / 1e6:.2f} MB")


---
## LM Head: Hidden States → Logits

### What is the LM Head?
The **Language Model Head** (`model.lm_head`) is a single linear layer:

```
Linear(hidden_dim → vocab_size)
```

It projects each hidden state vector from `hidden_dim` dimensions into `vocab_size` dimensions.

### What are logits?
**Logits** are raw, unnormalized scores — one per vocabulary token.
A high logit for token ID `1234` means the model thinks token `1234` is likely to come next.

To convert logits into probabilities, we apply **softmax**:
```
P(token_i) = exp(logit_i) / Σ exp(logit_j)
```

We only care about the **last token's logits** — that's the prediction for what comes next.


In [ ]:

with torch.no_grad():
    # Pass the full hidden states through the LM head
    # hidden_states: [1, seq_len, hidden_dim]
    # logits:        [1, seq_len, vocab_size]
    logits = model.lm_head(hidden_states)

print("=" * 55)
print("LM Head Output: Logits")
print("=" * 55)

print(f"\n📐 logits.shape = {logits.shape}")
print( "   Breakdown:")
print(f"     batch_size = {logits.shape[0]}")
print(f"     seq_len    = {logits.shape[1]}")
print(f"     vocab_size = {logits.shape[2]:,}   ← score for every possible next token")

# We only need the LAST position's logits for next-token prediction
last_token_logits = logits[0, -1, :]   # shape: [vocab_size]
print(f"\n🎯 Last-position logits shape: {last_token_logits.shape}")

# Convert to probabilities via softmax
probs = torch.softmax(last_token_logits, dim=-1)

# Top 5 most likely next tokens
top5_probs, top5_ids = torch.topk(probs, k=5)

print(f"\n📊 Top-5 next token predictions:")
print(f"   {'Rank':>4}  {'Token ID':>9}  {'Token':>12}  {'Probability':>12}")
print(f"   {'─'*4}  {'─'*9}  {'─'*12}  {'─'*12}")
for rank, (token_id, prob) in enumerate(zip(top5_ids, top5_probs), 1):
    token_str = tokenizer.decode([token_id.item()])
    print(f"   {rank:>4}  {token_id.item():>9}  {repr(token_str):>12}  {prob.item():>11.4%}")

print(f"\n💡 The model has assigned a probability to each of the {logits.shape[2]:,} tokens.")
print( "   Sum of all probs (should be 1.0):", round(probs.sum().item(), 4))


---
##  Argmax: Greedy Decoding Step-by-Step

### What is Greedy Decoding?
The simplest decoding strategy: **always pick the token with the highest logit**.

```python
next_token_id = argmax(logits[last_position])
```

**Pros:** Fast, deterministic, reproducible.
**Cons:** Can get stuck in repetitive loops; never explores alternative continuations.

### The autoregressive loop
To generate a full sentence, we repeat this process:
```
Step 1: [The, capital, of, France, is]  → predict: 'Paris'
Step 2: [The, capital, of, France, is, Paris]  → predict: '.'
Step 3: [The, capital, of, France, is, Paris, .]  → predict: '<EOS>'
```
Each new token gets **appended** to the input, and we run another forward pass.
This is why it's called **autoregressive** generation.


In [ ]:

print("=" * 55)
print("Greedy Decoding — Step by Step")
print("=" * 55)

# --- Single-step greedy pick ---
print("\n▶ Step 1: Pick the highest-logit token (argmax)")
next_token_id = torch.argmax(last_token_logits)      # scalar tensor
next_token_str = tokenizer.decode([next_token_id.item()])

print(f"   argmax token ID : {next_token_id.item()}")
print(f"   decoded token   : '{next_token_str}'")

# --- Multi-step greedy loop (manual) ---
print("\n▶ Step 2: Autoregressive loop (manual, 10 new tokens)")
print(f"   Starting prompt: \"{prompt}\"\n")

current_ids = input_ids.clone()  # shape: [1, seq_len]

generated_tokens = []
MAX_NEW_TOKENS = 10

with torch.no_grad():
    for step in range(MAX_NEW_TOKENS):
        # Full forward pass through model
        outputs = model(input_ids=current_ids)

        # Logits for the very last token position
        step_logits = outputs.logits[0, -1, :]          # [vocab_size]

        # Greedy pick
        next_id = torch.argmax(step_logits).unsqueeze(0).unsqueeze(0)  # [1,1]
        next_str = tokenizer.decode([next_id.item()])

        print(f"   Step {step+1:02d}: token_id={next_id.item():<7} → '{next_str}'")

        generated_tokens.append(next_id.item())

        # Stop early if EOS token is produced
        if next_id.item() == tokenizer.eos_token_id:
            print("   🛑 EOS token produced — stopping.")
            break

        # Append new token to sequence
        current_ids = torch.cat([current_ids, next_id], dim=1)  # [1, seq_len+1]

full_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(f"\n✅ Generated continuation: '{full_output}'")
print(f"   Full text: '{prompt}{full_output}'")


---
##  Pipeline: High-Level API vs Manual Inference

### What is `pipeline()`?
Hugging Face's `pipeline()` wraps **tokenizer + model + decoding** into a single function call.
It's the "batteries-included" version — great for quick experiments, not for understanding internals.

| Aspect | `pipeline()` | Manual Inference |
|---|---|---|
| Lines of code | 2 | 20+ |
| Transparency | Black box | Full visibility |
| Customization | Limited | Complete control |
| Learning value | Low | High |
| Production use | Rapid prototyping | Research / fine-tuning |


In [ ]:

# Build the pipeline — it reuses the model/tokenizer we already loaded
gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("=" * 55)
print("Pipeline API (High-Level)")
print("=" * 55)

pipe_result = gen_pipeline(
    prompt,
    max_new_tokens=20,
    do_sample=False,          # greedy, for fair comparison
    return_full_text=False,   # only return the NEW tokens
)

pipe_text = pipe_result[0]["generated_text"]
print(f"\nPrompt:    '{prompt}'")
print(f"Pipeline → '{pipe_text}'")

print("\n" + "=" * 55)
print("What pipeline() does internally (pseudocode):")
print("=" * 55)
print("""
1. inputs = tokenizer(text, return_tensors='pt')
2. input_ids → GPU
3. output_ids = model.generate(
       input_ids,
       attention_mask,
       max_new_tokens=...,
       do_sample=...,
       ...decoding config...
   )
4. text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
5. return [{"generated_text": text}]

→ All 10 steps we did manually, compressed into 1 call.
""")


---
## Decoding Strategies Explained

How do we pick the next token from the logit distribution?
This choice **dramatically** affects output quality.

| Strategy | Mechanism | Effect |
|---|---|---|
| **Greedy** | `argmax(logits)` | Always most likely; can be repetitive |
| **Sampling** | `random.choice(tokens, p=softmax(logits))` | Diverse; can be incoherent |
| **Temperature** | `logits / T` before softmax | `T<1` = sharper / more confident; `T>1` = flatter / more random |
| **Top-K** | Sample only from top K tokens | Prevents low-probability token selection |
| **Top-P (nucleus)** | Sample from smallest set with cumulative prob ≥ P | Adaptive, context-sensitive |


In [ ]:
# ── Greedy Decoding ──────────────────────────────────────────────────

# do_sample=False means: always pick the highest probability token
# deterministic — run it 100 times, get same result

test_prompt = "Once upon a time in a land far away"

def generate(prompt_text, **kwargs):
    """Helper: tokenize → generate → decode."""
    enc = tokenizer(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=40,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    # Slice off the input tokens, decode only the new ones
    new_tokens = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("🟢 Greedy Decoding  (do_sample=False)")
print("   Rule: always pick the single most probable next token.")
result_greedy = generate(test_prompt, do_sample=False)
print(f"   Prompt:  '{test_prompt}'")
print(f"   Output:  '{result_greedy}'\n")

# Run twice to prove it's deterministic
result_greedy2 = generate(test_prompt, do_sample=False)
print(f"   Run again (should be identical): '{result_greedy2}'")
print(f"   Deterministic: {result_greedy == result_greedy2} ✅")


In [ ]:
# ──  Sampling (do_sample=True) ────────────────────────────────────────

# do_sample=True means: draw the next token randomly from the probability distribution
# Non-deterministic — each run gives different output

print("🎲 Pure Sampling  (do_sample=True)")
print("   Rule: sample next token proportional to softmax(logits).")
print("   Non-deterministic — notice different outputs each run.\n")

for run in range(3):
    result = generate(test_prompt, do_sample=True)
    print(f"   Run {run+1}: '{result}'")


In [ ]:
# ─ Temperature ──────────────────────────────────────────────────────

# Temperature T scales logits BEFORE softmax:
#   logits_scaled = logits / T
#
#   T < 1.0  →  logits spread wider  →  high-prob tokens get even higher
#               → more deterministic / "confident"
#
#   T > 1.0  →  logits compressed closer together
#               → more uniform distribution → more random / "creative"
#
#   T = 1.0  →  no change (default softmax)

print("🌡️  Temperature Scaling\n")
temps = [0.1, 0.7, 1.0, 1.5]

for T in temps:
    result = generate(test_prompt, do_sample=True, temperature=T)
    label = "(very focused)" if T < 0.5 else "(focused)" if T < 1 else "(neutral)" if T == 1 else "(creative/random)"
    print(f"   T={T}  {label}")
    print(f"         '{result}'\n")


In [ ]:
# ── Top-K Sampling ───────────────────────────────────────────────────

# Top-K: at each step, keep only the K highest-probability tokens.
# Set all other logits to -infinity (zero probability).
# Then sample from the remaining K tokens.
#
# This prevents the model from accidentally picking bizarre low-prob tokens.
#
# Typical values: K=10 to K=100
# K=1 is equivalent to greedy decoding

print("🔝 Top-K Sampling\n")
k_values = [1, 10, 50]

for k in k_values:
    result = generate(test_prompt, do_sample=True, top_k=k)
    label = "(= greedy)" if k == 1 else "(restricted vocab)" if k < 20 else "(broader vocab)"
    print(f"   top_k={k:<4} {label}")
    print(f"           '{result}'\n")


In [ ]:
#  Top-P (Nucleus) Sampling ─────────────────────────────────────────

# Top-P (nucleus sampling):
# Sort tokens by probability, then keep the SMALLEST SET whose
# cumulative probability reaches P.
#
# Example with p=0.9:
#   token A: 60%  ← include
#   token B: 20%  ← include (cumsum = 80%)
#   token C: 10%  ← include (cumsum = 90%) ← stop here
#   token D: 5%   ← excluded
#   ...           ← all excluded
#
# Adaptive: uses fewer tokens when one token dominates,
#            uses more tokens when distribution is flat.
# This is the default strategy in most production chatbots.

print("🎯 Top-P (Nucleus) Sampling\n")
p_values = [0.5, 0.9, 0.99]

for p in p_values:
    result = generate(test_prompt, do_sample=True, top_p=p, top_k=0)  # top_k=0 disables top-k filter
    label = "(focused nucleus)" if p < 0.7 else "(balanced)" if p < 0.95 else "(wide nucleus)"
    print(f"   top_p={p}  {label}")
    print(f"            '{result}'\n")

print("🔥 Production tip: combine top_p=0.9 + temperature=0.7 for best results.")
result_combo = generate(test_prompt, do_sample=True, top_p=0.9, temperature=0.7)
print(f"   Combined: '{result_combo}'")


---
## Mini Chat Demo (Interactive Loop)

Phi-3-instruct uses a specific **chat template**:

```
<|user|>
Hello, how are you?<|end|>
<|assistant|>
```

The tokenizer has a built-in `apply_chat_template()` method that formats conversation history into this structure automatically.

We maintain a **message list** that grows with each turn — this gives the model "memory" of the conversation.

> 💡 Type `quit` or `exit` to end the conversation.


In [ ]:

def chat_with_phi3(
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
):
    """
    Multi-turn chat loop with Phi-3-mini-instruct.

    Maintains full conversation history in `messages` list so the model
    has context from previous turns (simulated memory via long context).
    """
    print("╔══════════════════════════════════════════╗")
    print("║   Phi-3 Mini Chat  —  type 'exit' to quit  ║")
    print("╚══════════════════════════════════════════╝")
    print(f"   Settings: temp={temperature}, top_p={top_p}, max_tokens={max_new_tokens}")
    print()

    # Conversation history — grows with each turn
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    turn = 0
    while True:
        # Get user input
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Chat ended]")
            break

        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit", "bye"):
            print("Phi-3: Goodbye! 👋")
            break

        # Append user turn to history
        messages.append({"role": "user", "content": user_input})

        # Format into Phi-3 chat template
        # apply_chat_template handles the <|user|>...<|end|><|assistant|> structure
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # adds <|assistant|> at end to prompt the model
        )

        # Tokenize the entire conversation so far
        inputs = tokenizer(formatted, return_tensors="pt").to(device)
        input_len = inputs["input_ids"].shape[1]

        # Generate response
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode only the new tokens (slice off the input)
        new_token_ids = output_ids[0][input_len:]
        response = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

        # Append assistant response to history (so model sees it next turn)
        messages.append({"role": "assistant", "content": response})

        turn += 1
        print(f"\nPhi-3: {response}")
        print(f"\n[Turn {turn} | Context tokens: {inputs['input_ids'].shape[1]} | New tokens: {len(new_token_ids)}]")
        print()

    return messages


# Start the chat!
conversation_history = chat_with_phi3(
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
)


---
##  Full Architecture Summary

Let's print the complete model architecture so you can see every layer.


In [ ]:

print("=" * 70)
print("Phi-3 Mini — Full Architecture")
print("=" * 70)

# Top-level structure
print("\n▶ Top-level components:")
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"   {name:<15} — {type(module).__name__:<30} ({params:.1f}M params)")

# Drill into one transformer block
print("\n▶ One Transformer Block (layers[0]):")
block = model.model.layers[0]
for name, module in block.named_children():
    print(f"   {name:<25} — {type(module).__name__}")

# Key dimensions
cfg = model.config
print("\n▶ Key Hyperparameters:")
attrs = [
    ("hidden_size",           "Hidden dimension (d_model)"),
    ("num_hidden_layers",     "Number of transformer blocks"),
    ("num_attention_heads",   "Attention heads"),
    ("num_key_value_heads",   "Key/Value heads (Grouped-Query Attention)"),
    ("intermediate_size",     "MLP intermediate dimension"),
    ("vocab_size",            "Vocabulary size"),
    ("max_position_embeddings","Max context length"),
]
for attr, label in attrs:
    if hasattr(cfg, attr):
        val = getattr(cfg, attr)
        print(f"   {label:<42}: {val:>8,}")

print("\n📌 GQA Note: num_key_value_heads < num_attention_heads")
print("   → multiple query heads share the same K/V heads")
print("   → reduces KV-cache memory without hurting quality")


---
## 🗺️ Final Summary — The Complete Journey

```
"The capital of France is"
           │
           ▼
    ┌─────────────┐
    │  Tokenizer  │  Text → Integer IDs
    │  (BPE/SPM)  │  shape: [1, seq_len]
    └──────┬──────┘
           │ input_ids
           ▼
    ┌─────────────────────────┐
    │   Embedding Layer       │  IDs → Dense Vectors
    │   [vocab_size, d_model] │  shape: [1, seq_len, 3072]
    └──────────┬──────────────┘
               │
               ▼
    ┌────────────────────────────────────────┐
    │  32 × Transformer Block               │
    │  ┌──────────────────────────────────┐ │
    │  │ RMSNorm                          │ │
    │  │ Grouped-Query Attention (GQA)    │ │
    │  │  → RoPE positional encoding      │ │
    │  │  → causal mask (no future peek)  │ │
    │  │ Residual connection              │ │
    │  │ RMSNorm                          │ │
    │  │ SwiGLU MLP (gate × up → down)   │ │
    │  │ Residual connection              │ │
    │  └──────────────────────────────────┘ │
    └────────────────┬───────────────────────┘
                     │
                     │ hidden_states: [1, seq_len, 3072]
                     ▼
    ┌─────────────────────────────┐
    │  LM Head (Linear)           │  d_model → vocab_size
    │  [3072 → 32064]             │  shape: [1, seq_len, 32064]
    └──────────────┬──────────────┘
                   │ logits (raw scores)
                   ▼
    ┌───────────────────────────────┐
    │  Decoding Strategy            │
    │  • Greedy: argmax             │
    │  • Sampling: random(p=probs)  │
    │  • Temperature / Top-K / Top-P│
    └──────────────┬────────────────┘
                   │ next_token_id
                   ▼
           Append → repeat loop
                   │
                   ▼
    ┌──────────────────────┐
    │  Tokenizer.decode()  │  IDs → Text
    └──────────────────────┘
                   │
                   ▼
           "Paris, France."
```

### You've now seen every step from the inside. 